# 00 はじめに — 基礎知識と用語から始めよう（異常検知）

> 🔒 すべて**合成データ**です。⏱ 所要時間: 読む 30分 ＋ 実行 数秒。**難しければ、「図の見方」と「用語集」だけ読んで、次のNotebookへ進んでください。**

予備知識ゼロから始めます。数式はほとんど使いません。分からない言葉が出たら、ここに戻ってきてください。

## 1. このプロジェクトは何をするの？（ひとことで）
**太陽光発電所の毎日の発電量から、「いつもと違う日」（故障・停電・影・パネルの劣化など）を自動で見つける**仕組みを作ります。これを**異常検知**と呼びます。

たとえ話: 健康診断で、いつもの体温・血圧から外れた日に「要注意」と知らせる体温計のようなものです。

難しいポイントは3つです。
1. **発電量が少ない＝異常、ではない。** 雨の日は発電量が少なくて当たり前。「天気のわりに少ない」ときだけが異常です。
2. **異常の正解（ラベル）は、現実にはほとんど無い。** 「この日が故障だった」と記録している人は少ない。
3. **警報を出しすぎると、誰も信用しなくなる。** 見逃しと誤警報のバランスが大事です。

## 2. 基礎知識

### 2-1. 時系列と「ふつう」
毎日の発電量は、**季節の波 ＋ 天気によるぶれ**でできています。「ふつう」の範囲を知るには、**平均**と**ばらつき**（標準偏差）を使います。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = ["Hiragino Sans", "AppleGothic", "DejaVu Sans"]

rng = np.random.default_rng(0)
days = np.arange(200)
normal = 10 + rng.normal(0, 1.0, 200)               # ふだんの発電量（平均10、ばらつき1）
data = normal.copy(); data[120:124] = [2.0, 1.0, 1.5, 2.5]     # 4日間、故障で急に落ちた（異常）
mean, sd = normal[:100].mean(), normal[:100].std()  # 最初の100日（正常）から「ふつう」を学ぶ
plt.figure(figsize=(11, 3.2)); plt.plot(days, data, lw=0.9)
plt.axhline(mean, color="g", label="平均（ふつう）"); plt.axhline(mean - 3 * sd, color="r", ls="--", label="平均 − 3×ばらつき（これより下は異常とする）")
plt.legend(); plt.xlabel("日"); plt.ylabel("発電量"); plt.grid(alpha=0.3); plt.show()
print(f"平均={mean:.2f}, 標準偏差={sd:.2f}, しきい値={mean - 3 * sd:.2f}")

**図の見方**: 緑の線が「ふつう」（平均）、赤の破線が**しきい値**です。日々の値は平均の周りで小さく揺れますが、120日目付近の4日間だけ、赤線を**大きく下回って**います。これが異常です。
**標準偏差**は「ふつうのばらつきの大きさ」です。「平均から標準偏差の3倍以上外れる」ことは、ふつうのデータではめったに起きません（約0.1%）。そこで、「3倍以上外れたら異常」とするのが、最も基本的な方法です。

### 2-2. 異常スコアとしきい値
**異常スコア**は「どれだけ異常っぽいか」を表す数字（大きいほど異常）。**しきい値**は「これを超えたら警報を出す」という境目です。
- しきい値を**低く**する → 小さな異常も拾える（**見逃しが減る**）が、正常な日にも警報が出る（**誤警報が増える**）。
- しきい値を**高く**する → 誤警報は減るが、見逃しが増える。
どちらが良いかは、運用次第（見逃しの損害 vs 誤警報で確認する手間）。

In [ ]:
score = np.abs(data - mean) / sd                    # 異常スコア：平均から何ばらつき分ずれたか
is_true = np.zeros(200, bool); is_true[120:124] = True
for th in [1.5, 2.0, 3.0, 5.0]:
    alarm = score > th
    fp = (alarm & ~is_true).sum(); tp = (alarm & is_true).sum()
    print(f"しきい値={th}: 警報 {alarm.sum():3d}日 / 正しい警報 {tp}日 / 誤警報 {fp:2d}日 / 見逃し {is_true.sum() - tp}日")

**出力の見方**: しきい値が1.5だと、4日の異常は全部拾えますが、正常な日にも24日の誤警報が出ます。しきい値を上げるほど誤警報は減り、3.0 で誤警報ゼロになります。この例の異常は大きい（発電量が約8割減）ので、5.0 でも全部拾えます。**もっと小さな異常**（例: 2割減る程度）だと、しきい値を上げた分だけ見逃します。この「誤警報と見逃しのトレードオフ」が、異常検知の核心です。

### 2-3. 評価の言葉
| 言葉 | 意味 | 例 |
|---|---|---|
| **誤警報** | 正常な日に出した警報 | 上の「誤警報 n日」 |
| **適合率** | 警報を出した日のうち、本当に異常だった割合 | 警報10日中3日が本物 → 0.3 |
| **再現率（検知率）** | 本当の異常のうち、警報を出せた割合 | 異常4日中3日 → 0.75 |
| **検知までの日数** | 異常が始まってから、最初の警報までの日数 | 0日なら即日 |
**精度（正解率）は使いません**: 異常は全体の数%しかないので、「常に正常と答える」だけで正解率95%になってしまうからです。

### 2-4. 天気を考慮する — 残差
「発電量が少ない」だけでは異常と言えません。**天気（日射量）から予想される発電量**を作り、**実際との差（残差）** を見ます。

In [ ]:
rng = np.random.default_rng(1)
sun = rng.uniform(3, 26, 200)                         # 日射量（日によって大きく変わる）
gen = 0.85 * sun * (1 + 0.03 * rng.normal(size=200))  # 発電量 ≒ 日射量×0.85
gen[150:154] *= 0.4                                   # 影で、4日間だけ6割減った（異常）
expected = 0.85 * sun                                 # 日射量から予想される発電量
resid = (expected - gen) / expected                   # 相対残差：期待より何割少ないか
fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
ax[0].plot(gen, lw=0.8); ax[0].set_title("発電量そのもの：天気で大きく揺れて、異常が目立たない"); ax[0].grid(alpha=0.3)
ax[1].plot(resid, lw=0.8, c="tab:red"); ax[1].set_title("期待との残差：異常の4日だけが突出する"); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

**図の見方**: 左は発電量そのもの。天気のせいで激しく上下し、150日付近の異常はそれに紛れています。右は**残差**（期待より何割少ないか）。天気の影響を取り除くと、普段は0付近で静か、異常の4日だけが大きく跳ね上がります。**期待値を作る**ことが、天気つきの異常検知のコツです。

### 2-5. 教師なし学習 — 正解ラベルなしで学ぶ
**教師あり**は「この日は異常」という正解付きのデータで学びます。しかし現実には正解がほとんど無い。そこで**教師なし**手法では、**正常なデータだけ**から「ふつう」を学び、そこから外れるものを異常とします。

- **Isolation Forest（孤立の木）**: データをランダムに区切っていき、**少ない区切りで1点だけ孤立する**点ほど異常、と考える。

In [ ]:
rng = np.random.default_rng(2)
pts = np.concatenate([rng.normal(10, 1, 49), [18.0]])      # 49個の普通の点と、1個の外れ値
def steps_to_isolate(p, rng, trials=200):
    n = []
    for _ in range(trials):
        lo, hi, cur, k = pts.min(), pts.max(), pts, 0
        while len(cur) > 1:
            cut = rng.uniform(cur.min(), cur.max())             # ランダムに区切る
            cur = cur[cur < cut] if p < cut else cur[cur >= cut]  # p が残るほうだけ見る
            k += 1
        n.append(k)
    return np.mean(n)
print("普通の点（10.2付近）を孤立させるのに要した区切り数 ≒", round(steps_to_isolate(pts[np.argmin(np.abs(pts - 10.2))], rng), 1))
print("外れ値（18.0）を孤立させるのに要した区切り数    ≒", round(steps_to_isolate(18.0, rng), 1))

**出力の見方**: 群れの中の普通の点は、何度も区切らないと1点にならない（区切り数が多い）のに対し、**離れた外れ値は、ほんの数回で孤立**します。**少ない区切りで孤立するほど異常**、が Isolation Forest の考え方です。

- **LOF**: 周りの点の混み具合を比べ、**周りより疎な点**を異常とみなす。
- **オートエンコーダ**: 入力を細い通路で**小さく圧縮**し、**元に戻す**ネットワーク。正常データだけで学ぶと、正常は上手に復元できる。異常は復元が下手（**復元誤差**が大きい）なので、異常と分かる。
  たとえ話: ふだん見慣れた風景だけを描く練習をした画家は、見慣れない風景を描くと下手になる。

### 2-6. 合成データと「注入」
現実には正解が無いので、このプロジェクトでは**正常な発電量をプログラムで作り、異常を自分で混ぜる（注入）**。異常の場所が分かるので、**答え合わせ**ができます。
ただし注意: **簡単な異常ばかり混ぜると、検知器が実力以上に見える**。異常の大きさを変えて「どこまで小さい異常を見つけられるか（**検知限界**）」を調べます。

### 2-7. データの分け方
**学習（train）**: 正常データだけで「ふつう」を学ぶ／**検証（val）**: しきい値や手法を選ぶ／**テスト（test）**: 最後に**1回だけ**評価。時間の順に分けます（未来を学習に使わない）。

## 3. 用語集（ここに戻ってきてください）

| 用語 | やさしい説明 |
|---|---|
| **異常検知** | 「いつもと違う」ものを見つけること。 |
| **正常データ** | 異常を含まない、普段どおりのデータ。 |
| **異常スコア** | 異常っぽさを表す数字。大きいほど異常。 |
| **しきい値** | これを超えたら警報を出す、という境目。 |
| **誤警報** | 正常な日に出してしまった警報。多いと信用されない。 |
| **適合率・再現率** | 警報の当たり率／異常のうち拾えた割合。 |
| **検知までの日数** | 異常が始まってから最初の警報までの日数。 |
| **イベント単位の検知率** | 異常イベント（例: 3日間の停電）のうち、期間中に1回でも警報が出た割合。**長い異常は偶然でも検知に数えられる**ので、「偶然の水準」も併記する。 |
| **偶然の水準** | 警報がでたらめに出ても、検知に数えられてしまう確率。基準として必ず見る。 |
| **共変量** | 発電量を説明する別の情報（日射量・日照率・天気）。 |
| **期待値・残差** | 天気から予想した「正常ならこのくらい」の値／実測との差。 |
| **平滑化** | 直近数日の平均を取ってノイズを減らすこと。ゆっくり進む劣化に効くが、異常が終わっても数日警報が残る（**余韻**）。 |
| **教師なし** | 正解ラベルを使わずに学ぶこと。 |
| **Isolation Forest／LOF** | 孤立しやすさ／周りより疎なこと、を見る教師なし手法。 |
| **オートエンコーダ・復元誤差** | 圧縮→復元するネット／元に戻せなかった度合い。 |
| **汚染** | 学習データに異常が混ざること。 |
| **注入** | 合成データに、わざと異常を混ぜること。 |
| **検知限界** | これより小さい異常は見つけられない、という境目。 |
| **異常の4種類** | **停電**（発電がほぼ止まる）、**劣化**（徐々に落ちる）、**影**（一定割合落ちる）、**固着**（値が動かなくなる）。 |
| **最終評価・予行演習** | 手法を全部決めた後に未使用データで1回だけ評価／その手順を別データで試すこと。 |

## 4. このプロジェクトの地図（読む順）

| Notebook | 問い | 一言 |
|---|---|---|
| 00 合成データ | 「少ない」と「異常」は違う？ | 違う（雨の日は少ない） |
| 01 ベースライン | AIなしの簡単な方法はどこで失敗する？ | 天気で誤警報が出る |
| 02 残差 | 天気を考慮すると？ | 誤警報が減る。余韻に注意 |
| 03 教師なし | 期待値モデルなしで見つけられる？ | 特徴量の作り方で変わる |
| 04 オートエンコーダ | 深層学習は役に立つ？ | 場面による。異常を覚えてしまう罠 |
| 05 組み合わせ・限界 | どこまで小さい異常を見つけられる？ | 検知限界がある |
| 06 実データ | 実データでも通用する？ | 通用しない部分がある |

## 5. 結果を読むときの心得（5つ）
1. **誤警報と見逃しを両方見る。** 片方だけ良くても意味がない。
2. **偶然の水準と比べる。**
3. **注入した異常が簡単すぎないかを疑う。**
4. **正常データだけで学び、しきい値は評価データと別のデータで決める。**
5. **合成データの結論を、そのまま現実に持ち込まない**（実データでは候補を出し、人が確認する）。

## 6. よくある疑問
- **Q. 深層学習（オートエンコーダ）が一番良いのでは？** A. 場面によります。簡単な「天気との残差」の方法が十分なことも多く、それを数字で確かめます。
- **Q. なぜ合成データ？** A. 異常の場所の正解が分かるので、検知器の良し悪しを測れるからです。
- **Q. 数式が出てきたら？** A. 「出力の見方」の文章だけ追えば大丈夫です。

次は **00_synthetic_data** へ進みましょう。